In [1]:
!pip install numpy


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
# Script to read ribbon data files
################################################
# each file is stored as "ribbon_X.txt"
# each file has *Nodes array of size n_nodesx3
# each file has the same *Triangles array of size n_trianglesx3 which tells which nodes make up each triangle and can be stored once
# read each file, extract the *Nodes array, append it to a big 2D array of size Nstepsx(n_nodesx3)

import numpy as np
import glob
import os

def read_ribbon_file(filename):
    nodes = []
    triangles = []

    mode = None
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith('*Nodes'):
                mode = 'nodes'
                continue
            elif line.startswith('*Triangles'):
                mode = 'triangles'
                continue

            if mode == 'nodes':
                nodes.append([float(x) for x in line.split(',')])
            elif mode == 'triangles':
                triangles.append([int(x) for x in line.split(',')])

    return np.array(nodes), np.array(triangles)


# --------------------------------------------------
# Read all ribbon files
# --------------------------------------------------

files = sorted(glob.glob("ribbon_*.txt"))
assert len(files) > 0, "No ribbon_*.txt files found"

nodes_all = []
triangles = None

for fname in files:
    nodes, tris = read_ribbon_file(fname)

    # flatten nodes immediately: (n_nodes, 3) -> (n_dof,)
    nodes_flat = nodes.reshape(-1)
    nodes_all.append(nodes_flat)

    if triangles is None:
        triangles = tris
    else:
        assert np.array_equal(triangles, tris), \
            f"Triangle connectivity mismatch in {fname}"

# Stack into (Nsteps, n_dof)
nodes_all = np.vstack(nodes_all)

print("nodes_all shape:", nodes_all.shape)      # (Nsteps, n_nodes*3)
print("triangles shape:", triangles.shape)

nodes_all shape: (11, 186)
triangles shape: (60, 3)


In [3]:
softRobot = lambda: None  # Create an empty object to hold data
softRobot.rod_edges = np.array([])
softRobot.face_nodes_shell = triangles - 1  # Convert to zero-based indexing
time_array = np.linspace(0, (nodes_all.shape[0]-1)*0.1, nodes_all.shape[0]).reshape(-1, 1)  # Assuming time step of 0.1s
print("time_array shape:", time_array)

time_array shape: [[0. ]
 [0.1]
 [0.2]
 [0.3]
 [0.4]
 [0.5]
 [0.6]
 [0.7]
 [0.8]
 [0.9]
 [1. ]]


In [4]:
import numpy as np

def map_node_to_dof(node_nums: int | np.ndarray) -> np.ndarray:
        return (3 * np.asarray(node_nums))[..., None] + np.array([0, 1, 2])
import numpy as np

def logDataForRendering(dofs, time_array, softRobot, Nsteps, static_sim, mapNodetoDOF):
    print(time_array.shape, dofs.shape)
    dof_with_time = np.hstack([time_array, dofs])
    print(np.shape(dof_with_time), np.shape(time_array), np.shape(dofs))
    n_rod_nodes = len(np.unique(softRobot.rod_edges))
    n_faces = len(softRobot.face_nodes_shell)
    print(np.shape(softRobot.face_nodes_shell))
    print(n_faces)

    if static_sim:
        rod_data = np.zeros((n_rod_nodes, 4))
        for j in range(n_rod_nodes):
            rod_data[j, 0] = dof_with_time[0, -1]
            rod_data[j, 1:] = dof_with_time[-1, 1 + mapNodetoDOF(j)]

        shell_data = np.zeros((3 * n_faces, 3))
        for j in range(n_faces):
            n1 = softRobot.face_nodes_shell[j, 0]
            n2 = softRobot.face_nodes_shell[j, 1]
            n3 = softRobot.face_nodes_shell[j, 2]
            shell_data[3*j:3*j+3, :] = np.vstack([
                dof_with_time[-1, 1 + mapNodetoDOF(n1)],
                dof_with_time[-1, 1 + mapNodetoDOF(n2)],
                dof_with_time[-1, 1 + mapNodetoDOF(n3)]
            ])

        np.savetxt('rawDataRod.txt', rod_data, fmt='%.6e')
        np.savetxt('rawDataShell.txt', shell_data, fmt='%.6e')
        return rod_data, shell_data

    # For dynamic case
    rod_data = np.zeros((n_rod_nodes * Nsteps, 4))
    for i in range(Nsteps):
        for j in range(n_rod_nodes):
            rod_data[i * n_rod_nodes + j, 0] = dof_with_time[i, 0]
            rod_data[i * n_rod_nodes + j, 1:] = dof_with_time[i, 1 + mapNodetoDOF(j)]

    shell_data = np.zeros((3 * n_faces * Nsteps, 3))
    for i in range(Nsteps):
        for j in range(n_faces):
            n1 = softRobot.face_nodes_shell[j, 0]
            n2 = softRobot.face_nodes_shell[j, 1]
            n3 = softRobot.face_nodes_shell[j, 2]
            idx = i * 3 * n_faces + 3 * j
            shell_data[idx:idx+3, :] = np.vstack([
                dof_with_time[i, 1 + mapNodetoDOF(n1)],
                dof_with_time[i, 1 + mapNodetoDOF(n2)],
                dof_with_time[i, 1 + mapNodetoDOF(n3)]
            ])

    np.savetxt('rawDataRod.txt', rod_data, fmt='%.6e')
    np.savetxt('rawDataShell.txt', shell_data, fmt='%.6e')

    return rod_data, shell_data

def export_rod_shell_data(robot, rod_file='rawDataRod.txt', shell_file='rawDataShell.txt',
                          rod_js='rodData.js', shell_js='shellData.js',
                          rod_radius=0.1, scaleFactor=100):
    """
    Export rod and shell data to .js files for visualization.

    Parameters
    ----------
    robot : object
        Object with attributes `rod_edges` and `face_nodes_shell`.
    rod_file : str
        Path to raw rod data (.txt).
    shell_file : str
        Path to raw shell data (.txt).
    rod_js : str
        Output JS file path for rod data.
    shell_js : str
        Output JS file path for shell data.
    rod_radius : float
        Radius of rods.
    scaleFactor : float
        Scale factor for coordinates.
    """

    # === Load rod data ===
    df = np.loadtxt(rod_file)
    n_rod_nodes = len(np.unique(robot.rod_edges))
    n_Tri = len(robot.face_nodes_shell)

    # Write rod data
    with open(rod_js, 'w') as fileID:
        fileID.write(f'nNodes = {n_rod_nodes};\n')
        fileID.write(f'rodRadius = {rod_radius};\n')
        fileID.write('nodesRod = [\n')

        for row in df:
            t, x, y, z = row
            x, y, z = x * scaleFactor, y * scaleFactor, z * scaleFactor
            fileID.write(f'{t}, 1, {x}, {y}, {z},\n')

        fileID.write(']\n;\n')

    # === Load shell data ===
    ds = np.loadtxt(shell_file)

    # Write shell data
    with open(shell_js, 'w') as shell_fileID:
        shell_fileID.write(f'nTri = {n_Tri},\n')
        shell_fileID.write('nodes = [\n')

        for row in ds:
            x, y, z = row * scaleFactor
            shell_fileID.write(f'{x}, {y}, {z},\n')

        shell_fileID.write('];\n')


In [5]:
logDataForRendering(nodes_all, time_array, softRobot, nodes_all.shape[0], static_sim=False, mapNodetoDOF=map_node_to_dof)
export_rod_shell_data(softRobot)

(11, 1) (11, 186)
(11, 187) (11, 1) (11, 186)
(60, 3)
60


/var/folders/z0/3frv2l990hb5ryd49z8vm4w80000gn/T/ipykernel_2646/461928485.py:87: UserWarning: loadtxt: input contained no data: "rawDataRod.txt"
  df = np.loadtxt(rod_file)
